# Entregable 2: artefactos persistentes y pilotos de ruido/potencia

**Preparación técnica opcional para una continuación posterior.** El cierre del informe utiliza los resultados disponibles; esta ruta no acredita pilotos nuevos ni exige ejecutarlos ahora.

Esta ruta corresponde al catálogo obligatorio de **16 sistemas** y al protocolo de tres anclas HPO. Usa **GPU CUDA real** y guarda los resultados en Google Drive. No sustituye el benchmark final ni el NAS completo de TiDE.

La comprobación inicial es el modo predeterminado: valida fuente, GPU y presupuesto, **sin abrir los CSV de desarrollo ni comenzar ajustes**. La ejecución está deshabilitada en esta entrega: faltan aceptación CUDA e integración revisada de overhead/caché con el presupuesto. Cambiar `EXECUTE_PILOTS` no habilita corridas. Una GPU conectada no demuestra que haya entrenamiento.

Si ya tienes la T4 conectada, puedes ejecutar la celda en esa misma VM una vez cerrado normalmente el controlador anterior. El lanzador detecta otros controladores locales; no los mata, no amplía el presupuesto y no reintenta automáticamente trabajos sin desenlace. Mantén cerradas otras sesiones de entrenamiento del proyecto.


## Preparar y, cuando sea admitido, ejecutar

1. Mantén la GPU conectada y espera el cierre normal del controlador anterior.
2. Ejecuta la celda. Autoriza el montaje de Drive si Colab lo solicita. El secreto `GITHUB_TOKEN`, si ya está configurado, se usa sólo para leer los diarios de GitHub y no se imprime.
3. La primera ejecución puede reutilizar o preparar el Python aislado 3.12.12 sin cambiar el kernel de Colab. Los archivos autorales se verifican por hash.
4. Mantén `EXECUTE_PILOTS=False`. Este apéndice prepara diagnósticos; la activación del ejecutor requiere otra versión verificada y no forma parte del cierre actual.

`STAGE="noise"` solicita los 60 ajustes del piloto TiDE. `STAGE="power"` prepara las 108 corridas HPO de los tres representantes y sus 15 ajustes de potencia. `"all"` solicita ambas etapas secuencialmente. Los límites científicos originales se conservan; las corridas pueden pausarse si no cabe otra reserva de 7.500 segundos.


In [ ]:
# Fuente y digest fijados por la publicación del lanzador; no usar main.
SOURCE_COMMIT = "__PILOT_SOURCE_COMMIT__"
LAUNCHER_SHA256 = "__PILOT_LAUNCHER_SHA256__"

STAGE = "all"             # "noise", "power" o "all"
EXECUTE_PILOTS = False     # True solicita ejecución; el presupuesto sigue siendo obligatorio.
OUTPUT = "/content/drive/MyDrive/DeepLearning_202630/E2/amended-pilots-v1"

import hashlib, json, re, urllib.request
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) or not re.fullmatch(r"[0-9a-f]{64}", LAUNCHER_SHA256):
    raise RuntimeError("Esta copia todavía no tiene un pin de publicación válido; abre el enlace publicado del notebook.")
url = ("https://raw.githubusercontent.com/jassan-bit/DeepLearning_202630/"
       + SOURCE_COMMIT + "/notebooks/run_e2_pilots_colab.py")
with urllib.request.urlopen(url, timeout=30) as response:
    source = response.read(131073)
if len(source) > 131072 or hashlib.sha256(source).hexdigest() != LAUNCHER_SHA256:
    raise RuntimeError("El lanzador no coincide con el archivo publicado; no se ejecutó.")
launcher = {"__name__": "e2_colab_pilot_launcher"}
exec(compile(source, "run_e2_pilots_colab.py", "exec"), launcher)
result = launcher["run_colab_pilots"](
    SOURCE_COMMIT, stage=STAGE, execute=EXECUTE_PILOTS, output=OUTPUT)
print(json.dumps(result, ensure_ascii=False, indent=2))


## Leer el estado y conservar los artefactos

- **Código 0 en preparación:** la admisión pasó; si `EXECUTE_PILOTS=False` no hubo entrenamiento.
- **Código 3:** ejecución deshabilitada o reserva completa no admitida. Se guarda `preflight.json` con `ready=false`; la GPU puede estar conectada correctamente. Repetir la celda con el mismo saldo no añade tiempo.
- **Código 2:** una comprobación o una ejecución falló/bloqueó. Conserva los registros y revisa su motivo; no repitas intentos abiertos automáticamente.

En Drive quedan el manifiesto de ejecución, las comprobaciones fechadas, diarios, predicciones, checkpoints, curvas por época, fallos y costes observados. El código fuente y la preparación se pueden reconstruir por su commit; los datos y fuentes descargados se conservan en `amended-pilots-assets`, fuera de la carpeta científica del controlador.

**k** sólo procede del piloto de ruido completo y válido; **R** sólo del piloto de potencia completo con sus criterios estadísticos. No se copian valores de ejemplos CPU ni se deducen de la conexión GPU o del contador histórico del plan anterior.

Si Colab se desconecta, los archivos de Drive permanecen, pero la VM y su proceso pueden desaparecer. Antes de continuar hay que conciliar cualquier intento iniciado sin cierre. Esta ruta no evita las desconexiones de Colab ni acredita por sí sola completar E2.
